In [ ]:
import pandas as pd
import icartt
import os
import warnings
import re
from datetime import datetime
import csv
from datetime import datetime, timedelta
from netCDF4 import Dataset
import numpy as np
from scipy import stats
import glob
from math import pi
import ast

# Read the CSV file
path_to_track = rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\MAC_DATASET_LLOD_FILTERED_10campaigns.csv"
df = pd.read_csv(path_to_track)

# Show all unique campaign values
unique_campaigns = df['Campaign'].unique()
print(f"All unique campaign values: {unique_campaigns}")

# Define the desired campaign order
campaign_order = [
    'FIREXAQ', 'SEAC4RS', 'CAMP2Ex', 'ASIA-AQ', 'DC3', 'DISCOVERAQ-DC', 
    'NAAMES(2016)', 'DISCOVERAQ-California', 'NAAMES(2017)', 'DISCOVERAQ-Texas', 
    'NAAMES(2015)', 'ACE-ENA', 'ISDAC', 'GOAMAZON', 'BBOP', 'ACMEV', 
    'CACTI', 'TCAP2013', 'TCAP2012', 'CARES', 'CALNEX', 'WECAN'
]

# Convert Date column from YYYYMMDD to datetime for processing
df['Date'] = pd.to_datetime(df['Date'], format='%Y%m%d')

# Group by Campaign and find min/max dates
date_ranges = df.groupby('Campaign')['Date'].agg(['min', 'max']).reset_index()

# Format dates as DD MONTH_NAME YYYY
date_ranges['Start_Date'] = date_ranges['min'].dt.strftime('%d %B %Y')
date_ranges['End_Date'] = date_ranges['max'].dt.strftime('%d %B %Y')

# Create a clean output with Campaign and formatted date ranges
result = date_ranges[['Campaign', 'Start_Date', 'End_Date']].copy()
result['Date_Range'] = result['Start_Date'] + ' - ' + result['End_Date']

# Reorder campaigns according to specified order
# Create a mapping for sorting
campaign_order_map = {campaign: i for i, campaign in enumerate(campaign_order)}
result['order'] = result['Campaign'].map(campaign_order_map)

# Sort by the order (campaigns not in the list will have NaN and appear at the end)
result = result.sort_values('order').drop('order', axis=1).reset_index(drop=True)

# Display results
print("\nDate ranges for each campaign (in specified order):")
print(result[['Campaign', 'Date_Range']])

print("\nDetailed date ranges (in specified order):")
print(result)

# Add detailed breakdown by Organization, Campaign, and Date
print("\n" + "="*80)
print("DETAILED BREAKDOWN BY ORGANIZATION, CAMPAIGN, AND DATE")
print("="*80)

# Create detailed breakdown
detailed_breakdown = df.groupby(['Organization', 'Campaign', 'Date']).size().reset_index(name='Row_Count')

# Calculate total rows per campaign for percentage calculations
campaign_totals = df.groupby(['Organization', 'Campaign']).size().reset_index(name='Total_Campaign_Rows')
detailed_breakdown = detailed_breakdown.merge(campaign_totals, on=['Organization', 'Campaign'])

# Calculate percentage within each campaign
detailed_breakdown['Percentage'] = (detailed_breakdown['Row_Count'] / detailed_breakdown['Total_Campaign_Rows'] * 100).round(2)

# Count unique dates per campaign
unique_dates_per_campaign = detailed_breakdown.groupby(['Organization', 'Campaign'])['Date'].nunique().reset_index(name='Unique_Dates')
detailed_breakdown = detailed_breakdown.merge(unique_dates_per_campaign, on=['Organization', 'Campaign'])

# Calculate campaign percentage of total dataset
total_rows = len(df)
detailed_breakdown['Campaign_Percentage'] = (detailed_breakdown['Total_Campaign_Rows'] / total_rows * 100).round(2)

# Calculate overall percentage for each individual date
detailed_breakdown['Overall_Percentage'] = (detailed_breakdown['Row_Count'] / total_rows * 100).round(4)

# Format Date back to YYYYMMDD for display
detailed_breakdown['Date'] = detailed_breakdown['Date'].dt.strftime('%Y%m%d')

# Sort by Organization, then by campaign order, then by date
org_campaign_order = []
for org in detailed_breakdown['Organization'].unique():
    for campaign in campaign_order:
        if campaign in detailed_breakdown[detailed_breakdown['Organization'] == org]['Campaign'].values:
            org_campaign_order.append((org, campaign))

# Create a sorting key
def get_sort_key(row):
    org = row['Organization']
    campaign = row['Campaign']
    try:
        org_campaign_idx = org_campaign_order.index((org, campaign))
    except ValueError:
        org_campaign_idx = 999  # Put unmatched at end
    return (org_campaign_idx, row['Date'])

detailed_breakdown['sort_key'] = detailed_breakdown.apply(get_sort_key, axis=1)
detailed_breakdown = detailed_breakdown.sort_values('sort_key').drop('sort_key', axis=1).reset_index(drop=True)

# Display the detailed breakdown
print("Detailed breakdown (comma-separated for Excel):")
print("Organization,Campaign,Date,Row_Count,Total_Campaign_Rows,Percentage,Unique_Dates,Campaign_Percentage,Overall_Percentage")
for _, row in detailed_breakdown.iterrows():
    print(f"{row['Organization']},{row['Campaign']},{row['Date']},{row['Row_Count']},{row['Total_Campaign_Rows']},{row['Percentage']},{row['Unique_Dates']},{row['Campaign_Percentage']},{row['Overall_Percentage']}")

print(f"\n{'='*80}")
print("SUMMARY BY ORGANIZATION AND CAMPAIGN (comma-separated for Excel)")
print("="*80)

# Also create a summary by Organization and Campaign
summary = detailed_breakdown.groupby(['Organization', 'Campaign']).agg({
    'Row_Count': 'sum',
    'Total_Campaign_Rows': 'first',
    'Unique_Dates': 'first',
    'Campaign_Percentage': 'first'
}).reset_index()

summary['sort_key'] = summary.apply(lambda row: org_campaign_order.index((row['Organization'], row['Campaign'])) 
                                   if (row['Organization'], row['Campaign']) in org_campaign_order else 999, axis=1)
summary = summary.sort_values('sort_key').drop('sort_key', axis=1).reset_index(drop=True)

print("Organization,Campaign,Row_Count,Total_Campaign_Rows,Unique_Dates,Campaign_Percentage")
for _, row in summary.iterrows():
    print(f"{row['Organization']},{row['Campaign']},{row['Row_Count']},{row['Total_Campaign_Rows']},{row['Unique_Dates']},{row['Campaign_Percentage']}")

All unique campaign values: ['CARES' 'ACMEV' 'TCAP2013' 'BBOP' 'CACTI' 'ASIA-AQ' 'DC3'
 'DISCOVERAQ-California' 'DISCOVERAQ-Texas' 'FIREXAQ' 'NAAMES(2015)'
 'NAAMES(2016)' 'NAAMES(2017)']

Date ranges for each campaign (in specified order):
                 Campaign                             Date_Range
0                 FIREXAQ          17 July 2019 - 29 August 2019
1                 ASIA-AQ       06 February 2024 - 27 March 2024
2                     DC3             18 May 2012 - 22 June 2012
3            NAAMES(2016)            01 June 2016 - 01 June 2016
4   DISCOVERAQ-California     16 January 2013 - 06 February 2013
5            NAAMES(2017)  12 September 2017 - 16 September 2017
6        DISCOVERAQ-Texas  06 September 2013 - 26 September 2013
7            NAAMES(2015)    12 November 2015 - 12 November 2015
8                    BBOP            15 July 2013 - 30 July 2013
9                   ACMEV     02 August 2015 - 04 September 2015
10                  CACTI    14 November 201